# ICE Enforcement Database — Quickstart

This notebook lets you query ICE enforcement data (arrests, detainers, detentions, removals, custody decisions) directly from Hugging Face — no download needed.

**17.8M+ rows** across 5 tables covering FY2012 through October 2025, from two FOIA releases combined and deduplicated.

Data from the [Deportation Data Project](https://law.ucla.edu/academics/centers/center-immigration-law-and-policy/deportation-data-project) (Berkeley Law / UCLA).

Source: [github.com/ian-nason/ice-database](https://github.com/ian-nason/ice-database)

## Setup

In [ ]:
!pip install -q duckdb

import duckdb

con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql("""
    ATTACH 'https://huggingface.co/datasets/Nason/ice-database/resolve/main/ice.duckdb'
    AS ice (READ_ONLY)
""")
con.sql("SELECT * FROM ice._metadata").show()

## Arrests by Month

In [ ]:
df = con.sql("""
    SELECT DATE_TRUNC('month', apprehension_date) AS month,
           COUNT(*) AS arrests
    FROM ice.arrests
    WHERE apprehension_date IS NOT NULL
    GROUP BY 1 ORDER BY 1
""").fetchdf()

df.set_index('month')['arrests'].plot(figsize=(14, 4), title='ICE Administrative Arrests by Month')

## Detention Duration Distribution

In [ ]:
con.sql("""
    SELECT
        CASE
            WHEN days <= 7 THEN '0-7 days'
            WHEN days <= 30 THEN '8-30 days'
            WHEN days <= 90 THEN '31-90 days'
            WHEN days <= 180 THEN '91-180 days'
            WHEN days <= 365 THEN '181-365 days'
            ELSE '365+ days'
        END AS bucket,
        COUNT(*) AS stays
    FROM (
        SELECT DATEDIFF('day', stay_book_in_date,
                        COALESCE(stay_book_out_date, CURRENT_DATE)) AS days
        FROM ice.detentions
        WHERE stay_book_in_date IS NOT NULL
    )
    GROUP BY 1
    ORDER BY MIN(days)
""").show()

## Top Citizenship Countries (Removals)

In [ ]:
df = con.sql("""
    SELECT citizenship_country, COUNT(*) AS removals
    FROM ice.removals
    WHERE citizenship_country IS NOT NULL
    GROUP BY 1 ORDER BY 2 DESC LIMIT 10
""").fetchdf()

df.plot.barh(x='citizenship_country', y='removals', figsize=(10, 5),
             title='Top 10 Citizenship Countries (Removals)', legend=False)

## Data Source Breakdown

Every row has a `data_source` column showing which FOIA release it came from.

In [ ]:
for table in ['arrests', 'detainers', 'detentions', 'removals', 'rca_decisions']:
    print(f"\n{table}:")
    con.sql(f"""
        SELECT data_source, COUNT(*) AS rows,
               MIN(CASE WHEN '{table}' = 'arrests' THEN apprehension_date
                        WHEN '{table}' = 'detentions' THEN stay_book_in_date
                        WHEN '{table}' = 'removals' THEN departure_date
                        ELSE NULL END)::DATE AS earliest,
               MAX(CASE WHEN '{table}' = 'arrests' THEN apprehension_date
                        WHEN '{table}' = 'detentions' THEN stay_book_in_date
                        WHEN '{table}' = 'removals' THEN departure_date
                        ELSE NULL END)::DATE AS latest
        FROM ice.{table}
        GROUP BY 1 ORDER BY 1
    """).show()